# 00 · Setup and inventory

**Run this first**, every time you start on a new data folder.

1. Checks that the Python environment is the right one.
2. Loads your config file (all paths live there, nowhere else).
3. Lists every take found in `data_dir`: start time, duration, number of channels, sample rate,
   split files.
4. Estimates how big the processed cache will be.

**What you edit:** only the cell marked `YOUR SETTINGS`. To start a real experiment, copy
`config/TEMPLATE.yaml` to `config/local_<experiment>.yaml`, fill it in, and point `CONFIG` at it.
The demo config works out of the box after `python -m zoom_acoustics demo demo_data`.

In [1]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"

In [2]:
import sys, importlib
print("python:", sys.executable)
print("       ", sys.version.split()[0])
for m in ["numpy", "scipy", "pandas", "matplotlib", "soundfile", "yaml", "openpyxl"]:
    mod = importlib.import_module(m)
    print(f"  {m:11s} {getattr(mod, '__version__', '?')}")

python: /home/tmittal/miniforge3/envs/claude-science-env/bin/python
        3.11.15
  numpy       2.4.3
  scipy       1.17.1


  pandas      3.0.2
  matplotlib  3.10.9
  soundfile   0.14.0
  yaml        6.0.3
  openpyxl    3.1.5


In [3]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

config   : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/config/demo.yaml
data_dir : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/audio
cache_dir: /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache
figures  : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/figures


## Takes found

A **take** is one continuous recording. The recorder may have written it as

* one multi-channel file, `260910_011.WAV`;
* several split files (the recorder starts a new file at about 2 GB), `260909_008_0001.WAV`, `_0002.WAV`, which are
  joined seamlessly;
* one mono file per track, `..._Tr1.WAV`, `..._Tr2.WAV`, which become the channels of one take.

`start` is the **recorder clock** time written into each file (1 s resolution). If the recorder clock
was wrong, set `recorder_clock_offset_s` in the config (see the student guide, *Clocks*).

In [4]:
takes = za.discover_takes(cfg.data_dir, recursive=cfg.recursive, pattern=cfg.file_pattern)
bad = cfg.unmatched_take_keys(takes)
if bad:
    print("WARNING: config 'takes:' entries that match no take (typo?):", bad)
table = za.takes_table(takes)
table

,take,start,duration_s,sr,n_channels,channels,tracks,n_parts,n_files,subtype,size_GB
0,DEMO_001,2026-10-01 10:00:00,60.0,48000,4,"1,2,3,4","Hyd_A,Hyd_B,Contact,AirMic",1,1,FLOAT,0.046
1,DEMO_002,2026-10-01 10:02:00,180.0,48000,4,"1,2,3,4","Hyd_A,Hyd_B,Contact,AirMic",1,1,FLOAT,0.138
2,DEMO_003,2026-10-01 10:06:00,150.0,48000,2,"1,2","Hyd_A,Hyd_B",2,2,FLOAT,0.058
3,DEMO_004,2026-10-01 10:10:00,120.0,48000,1,1,Hyd_A,1,1,FLOAT,0.023


## Channels

Channel numbers are the recorder **track numbers** (Ch1 = Tr1). Names and sensor types come from the
`channels:` section of the config. `sensor` must be one of `hydrophone`, `contact`, `air`, `other`.
Takes may have different channel sets. That is expected; every tool skips channels a take does not have.

In [5]:
all_ch = sorted({c for t in takes for c in t.channel_numbers})
pd.DataFrame([dict(channel=c, **cfg.channel_info(c),
                   in_takes=sum(c in t.channel_numbers for t in takes)) for c in all_ch])

,channel,name,sensor,in_takes
0,1,Hydrophone A,hydrophone,4
1,2,Hydrophone B,hydrophone,3
2,3,Contact sensor,contact,2
3,4,Air mic,air,2


## Processing plan and cache size

`cache_GB` is what processing will write to `cache_dir`. If it is large (long recordings), raise
`dsp.psd_dt` (e.g. 1 s or 5 s) or set `dsp.psd_fmax_hz` in the config; see the student guide, *Long recordings*.

In [6]:
rows = []
for t in takes:
    p = za.plan(t, cfg)
    rows.append(dict(take=t.name, channels=p["channels"], sr=t.sr,
                     freq_resolution_Hz=round(t.sr / p["nperseg"], 2),
                     store_envelope=p["store_env"], cache_GB=round(sum(p["size"].values()), 3),
                     notes="; ".join(p["notes"])))
pd.DataFrame(rows)

,take,channels,sr,freq_resolution_Hz,store_envelope,cache_GB,notes
0,DEMO_001,"[1, 2, 3, 4]",48000,23.44,True,0.006,
1,DEMO_002,"[1, 2, 3, 4]",48000,23.44,True,0.018,
2,DEMO_003,"[1, 2]",48000,23.44,True,0.007,
3,DEMO_004,[1],48000,23.44,True,0.003,
